# AirSense-R: Do Air-Quality Forecasters Keep Their Rank Under Sensor Degradation and Site Shift?

## 07 Statistical and failure analysis

**Permitted partitions: saved VALIDATION and FINAL TEST — READ ONLY.**

Choose a CPU runtime and **Run all**. Authorize the original Drive account and GitHub secret. No SHA, stage or run name is required. This verifies P7 prediction files, final native model bytes, validation selection, metrics and independent block-bootstrap intervals before failure analysis. It then checks saved P6 arrays and reports concentration bands, seasons, worst errors and rank stability. It never trains, predicts again or downloads raw data. Allow substantial Drive reading time; completed units publish incrementally to one analysis branch.

Concentration bands and seasonal/worst-case analyses are **post-results exploratory diagnostics**, not health categories or additional confirmatory hypotheses. Results remain within-Beijing evidence. After disconnection rerun cell 3; if the runtime was replaced, Run all. Never delete locks. Use only one runtime. Scientific acceptance follows review of the completed evidence.

## 1. Setup

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY = "https://github.com/Engr-Daniel/airsense-r.git"
# Use the full commit SHA of the GitHub revision containing this notebook.
# Resume using the SAME revision and run ID as the original execution.
TEST_MODE = os.environ.get("AIRSENSE_NOTEBOOK_TEST_MODE") == "1"
if TEST_MODE:
    ROOT = Path(os.environ["AIRSENSE_NOTEBOOK_TEST_ROOT"]).resolve()
else:
    CODE_REVISION = "af76b79578036cfeb8fb614db086975bc88173b8"
    if len(CODE_REVISION) != 40 or any(c not in "0123456789abcdef" for c in CODE_REVISION.lower()):
        raise ValueError("Use a full Git commit SHA, not a moving branch name")
    ROOT = Path.cwd() / "airsense-r-p8-af76b7957803"
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(ROOT)], check=True)
        subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", CODE_REVISION], check=True)
    else:
        remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
        head = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
        problems = []
        if remote != REPOSITORY:
            problems.append("origin is not the expected repository")
        if head != CODE_REVISION:
            problems.append(f"checkout is {head}, requested {CODE_REVISION}")
        if dirty:
            problems.append("checkout has modified/untracked files")
        if problems:
            raise RuntimeError("; ".join(problems) + ". Preserve existing work and use a fresh runtime filesystem; restarting only the kernel does not remove the checkout.")
    if sys.version_info < (3, 11):
        raise RuntimeError("Python 3.11+ required")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[notebooks]"], check=True)
    print("Code revision:", CODE_REVISION)
# Editable installs may not activate their .pth file in an already-running kernel.
# Activate only this checkout, and reject cached modules from another checkout.
import importlib
SOURCE_DIR = (ROOT / "src").resolve()
PACKAGE_DIR = SOURCE_DIR / "airsense_r"
if not (PACKAGE_DIR / "__init__.py").is_file():
    raise RuntimeError("Selected checkout has no airsense_r package")
for module_name, module in tuple(sys.modules.items()):
    if module_name == "airsense_r" or module_name.startswith("airsense_r."):
        location = getattr(module, "__file__", None)
        if location is None or not Path(location).resolve().is_relative_to(PACKAGE_DIR):
            raise RuntimeError("Another airsense_r checkout is already imported; restart the runtime")
source_path = str(SOURCE_DIR)
if source_path in sys.path:
    sys.path.remove(source_path)
sys.path.insert(0, source_path)
importlib.invalidate_caches()
import airsense_r
if Path(airsense_r.__file__).resolve() != PACKAGE_DIR / "__init__.py":
    raise RuntimeError("Imported package does not match the selected checkout")
print("Workspace:", ROOT)
print("Verified package:", airsense_r.__file__)

from airsense_r.artifacts import source_hash
if source_hash(ROOT) != "00975004069736d0faefa4d6f5d4de431a74f61e911925b5dbe4354972545721":
    raise RuntimeError("Notebook/source mismatch; reopen the matching published notebook")


## 2. Connect original saved evidence

In [ ]:
from airsense_r.colab import configure_github_secret
RUN_ID = "p8-analysis-01"
if TEST_MODE:
    MODEL_DIRECTORY = Path(os.environ["AIRSENSE_P8_TEST_MODELS"])
    RUN_ID = os.environ["AIRSENSE_P8_TEST_RUN"]
    REMOTE = "fixture"
else:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    MODEL_DIRECTORY = Path("/content/drive/MyDrive/AirSense-R/model-checkpoints")
    configure_github_secret(ROOT, userdata.get("AIRSENSE_GITHUB_TOKEN"))
    REMOTE = REPOSITORY
if not (MODEL_DIRECTORY / "storage.json").is_file():
    raise FileNotFoundError("Original Drive storage is missing; check the mounted account")
JOB_DIRECTORY = ROOT / ".tmp/notebook-jobs" / RUN_ID
print("Ready: verify saved P7 predictions, then analyze P6/P7 failures. No training.")


## 3. Verify, analyze and save incrementally

In [ ]:
from airsense_r.notebook_job import run_and_watch
command = [sys.executable, str(ROOT / "scripts/run_failure_analysis.py"),
           "--model-directory", str(MODEL_DIRECTORY), "--remote", REMOTE]
if TEST_MODE:
    import json
    command = json.loads(os.environ["AIRSENSE_P8_TEST_COMMAND"])
run_and_watch(command, JOB_DIRECTORY, ROOT)


## 4. Review completion

In [ ]:
import json
from airsense_r.artifacts import RunStore, source_hash
from airsense_r.notebook_job import status
if status(JOB_DIRECTORY)["state"] != "complete":
    raise RuntimeError("Analysis is not complete; rerun cell 3 to reconnect or inspect its error")
directory = ROOT / ".tmp/checkpoint-checkouts" / RUN_ID / "results" / RUN_ID
manifest = json.loads((directory / "manifest.json").read_bytes())
store = RunStore(directory, manifest["provenance"])
if manifest["provenance"].get("source_hash") != source_hash(ROOT) or manifest["provenance"].get("run_id") != RUN_ID:
    raise RuntimeError("Analysis provenance mismatch")
if not store.load_json("p7-verification.json").get("passed") or not store.load_json("complete.json").get("execution_complete"):
    raise RuntimeError("Verification/analysis completion record missing")
display(store.load_json("p7-verification.json"))
display(store.load_json("dashboard-p8.json"))
print("P8 EXECUTION COMPLETE. Save this notebook with outputs for independent review.")
